# Pre-processing CCL Data

Here, I am providing the code that I used to get from the raw data sets to the standardized version that can then be used to fit the data using the hyperbolic function. The process is slightly different for each dataset, based on the way the data is presented in the source studies. All studies where data is already expressed as mean and standard deviation of full cycles do not need any further processing and are not included in this code.

The standardized versions of the raw data are here created separately and saved as ```pandas.DataFrames()```. When all datasets are standardized in this way, I combine them and save them as an excel file with a separate sheet for each study. I decided to present the code as a Jupyter Notebook for better readability and so that I can comment on the methods for each dataset.

As data was collected from different source studies, I do not provide the raw data sets here. Citations to the individual data sets can be found as comments in the code blocks. 

## Setup

In [ ]:
# necessary path specification so that the Jupyter Notebook can acces the utils and other folders 

from pathlib import Path
import sys

folder = Path.cwd().parent
sys.path.insert(0, str(folder))

data_path = "FILE_PATH"

# import of necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.pre_processing import convert_lineage_tree, add_SDs

In [5]:
# define the maximum generation for the cases in which they should be cut

max_generations = {
    'Zebrafish': 13,
    'Dreissena': 12,
}

In [14]:
# ZEBRAFISH
# only until including generation 13 
# std calculated as sqrt of the variance -> might need to check in with Nicoletta's group if that is the meaning of what they recorded as variance!


dfz = pd.read_excel(f'{data_path}Raw data zebrafish.xlsx')

dfz['species'] = 'Zebrafish'

# calculate the standard deviation from the variance given in the raw data 
dfz['std'] = np.sqrt(dfz['variance'])                         
dfz['unit'] = 'min'

# drop last datapoint because of potential underestimation 
dfz = dfz[dfz['generation'] <= max_generations['Zebrafish']]

dfz = dfz.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [16]:
dfa1 = pd.read_excel(f'{data_path}Raw data Axolotl 1 embryo.xlsx') 


In [17]:
# AXOLOTL 1 EMBRYO 

dfa1 = pd.read_excel(f'{data_path}Raw data Axolotl 1 embryo.xlsx') 

# calculate duration based on absolute timings -> for that first sort by 'Cell' (= arbitrary lineage, traces back to the mother cell) and by 'cleavage' for the generation
dfa1 = dfa1.sort_values(by = ['Cell', 'cleavage']) 
# within the data for one cell, calculate the difference between an x-value (position on time axis) and it's previous x-value (position t-1 on time axis)
# add this new value (= duration from one cell division to the next) to the dataframe in a column "Duration"
# this version (not explicit tree) is possible because except for gen 15 I have 13 datapoints in each generation 
dfa1['Duration'] = dfa1.groupby('Cell')['x'].diff()

# now there are some NaN-values in the data -> drop those because those would be the cell cycle duration for the "birth" of the respective first cells of an arbitrary lineage (no cycle yet)
dfa1 = dfa1.dropna()

# to caclulate the mean and std I group by cleavage (= generation s) and then use the pandas.Series.mean() and .std() commands
dfa1 = dfa1.groupby('cleavage')['Duration'].agg(['mean', 'std']).reset_index()


dfa1['species'] = 'Axolotl'
dfa1['generation'] = dfa1['cleavage']
dfa1['unit'] = 'min'

dfa1 = dfa1.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [18]:
# AXOLOTL 15 embryos

dfa2 = pd.read_excel(f'{data_path}Raw data Axolotl 15 embryos.xlsx') 

minute_conversion_factor = 80

dfa2['generation'] = dfa2['Cycle']
dfa2['mean'] = dfa2['Average time'] * minute_conversion_factor
dfa2['std'] = dfa2['SD'] * minute_conversion_factor
dfa2['species'] = 'Axolotl'
dfa2['unit'] = 'min'
dfa2['mean (a.u.)'] = dfa2['Average time']
dfa2['std (a.u.)'] = dfa2['SD']
dfa2[f'unit (original a.u. * {minute_conversion_factor} = min)']  = 'a.u.'


dfa2 = dfa2.reindex(columns=['species', 'generation', 'mean', 'std', 'unit', 'mean (a.u.)','std (a.u.)', f'unit (original a.u. * {minute_conversion_factor} = min)' ])

In [19]:
# DREISSENA

# Dreissena is the first dataset that appears here where I have a cell lineage tree and need to specify the individual mother cells so that the duration is calculated by substracting the correct values.
dfd = pd.read_excel(f'{data_path}Raw data Dreissena.xlsx')

# sort dataframe by cell and cleavage (for calculation of duration of cell cycles) -> within one cycle, sort by y (position within the lineage tree) 
dfd = dfd.sort_values(by = ['Generation', 'y']).reset_index(drop = True)

dfd = dfd[dfd['Generation'] <= 6]

durations = [0]
for i in range(1,7):
    #print(i)
    
    current_cleaveage = dfd[dfd['Generation'] == i].reset_index()
    previous_cleavage = dfd[dfd['Generation'] == i-1].reset_index()

    
    for j in range(len(current_cleaveage)):
        current_division_time = current_cleaveage['x'][j]
        last_division = current_cleaveage['mother cell index'][j]
        last_division_time = previous_cleavage['x'][last_division]

        duration = (current_division_time - last_division_time)*60
        durations.append(duration)

dfd['Duration'] = durations
   
# # for this data, I create a new dataframe called df_stat that now just has the mean and standard deviation per cleavage 
dfd = dfd.groupby('Generation')['Duration'].agg(['mean', 'std']).reset_index()

dfd = dfd[dfd['Generation']>0]
dfd = dfd[dfd['Generation']<= max_generations['Dreissena']]

dfd['species'] = 'Dreissena'
dfd['generation'] = dfd['Generation']
dfd['unit'] = 'min'

# drop the first few entries (start from cleavage 6) -> bc data before that is not reliable 
# dfd = dfd[dfd['Cleavage'] > 1]
dfd = dfd.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

dfd

,species,generation,mean,std,unit
1,Dreissena,1,90.000000,NaN,min
2,Dreissena,2,62.142857,6.060915,min
3,Dreissena,3,57.857143,3.912304,min
4,Dreissena,4,67.500000,3.239695,min
5,Dreissena,5,130.044643,43.189559,min
6,Dreissena,6,169.285714,93.246675,min


In [ ]:
# DROSOPHILA 1 (Foe & Alberts, 1983)

dfd1 = pd.read_excel(f'{data_path}Raw data Drosophila.xlsx', sheet_name = 'Study 1') 

dfd1['generation'] = dfd1['Cycle']
dfd1['mean'] = dfd1['Mean Duration']
dfd1['std'] = dfd1['SD Duration']
dfd1['species'] = 'Drosophila'
dfd1['unit'] = 'min'

dfd1 = dfd1.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [ ]:
# DROSOPHILA 2 (Sung et al., 2013)

dfd2 = pd.read_excel(f'{data_path}Raw data Drosophila.xlsx', sheet_name ='Study 2') 

dfd2['generation'] = dfd2['Cycle']
dfd2['std'] = dfd2['SD (min)']
dfd2['mean'] = dfd2['Mean (min)']
dfd2['species'] = 'Drosophila'
dfd2['unit'] = 'min'

dfd2 = dfd2.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [ ]:
# DROSOHILA 3 (Henry et al., 2022)

# no pre-processing necessary, already saved under the correct format
dfd3 = pd.read_excel(f'{data_path}Raw data Drosophila.xlsx', sheet_name ='Study 3') 

In [ ]:
# XENOPUS 1 (Satoh, 1977)

dfx1 = pd.read_excel(f'{data_path}Raw data Xenopus.xlsx', sheet_name ="Study 1") 

# Cleavage counted differently 
dfx1['generation'] = (dfx1['Cleavage'] - 1)
dfx1['mean'] = dfx1['Average']
dfx1['std'] = dfx1['SD']
dfx1['species'] = 'Xenopus'
dfx1['unit'] = 'min'

dfx1 = dfx1.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [ ]:
# XENOPUS 2 (Clute & Masui, 1995)

dfx2 = pd.read_excel(f'{data_path}Raw data Xenopus.xlsx', sheet_name = "Study 2") 

dfx2['generation'] = dfx2['Cleavage']
dfx2['mean'] = dfx2['Average']
dfx2['std'] = dfx2['Coefficient of variation']*dfx2['Average']
dfx2['species'] = 'Xenopus'
dfx2['unit'] = 'min'
dfx2['CoV'] = dfx2['Coefficient of variation']

dfx2 = dfx2.reindex(columns=['species', 'generation', 'mean', 'std', 'unit', 'CoV'])


In [ ]:
# NEMATOSTELLA

dfn = pd.read_excel(f'{data_path}Raw data Nematostella.xlsx') 

# get discretized generation from the x-position in the raw data (from image with PlotDigitizer)
generations = []
for i in range(len(dfn)):
    generations.append(int(round(dfn['x'][i])))
dfn['generation'] = generations

# get mean and std for the defined generations
dfn = dfn.groupby('generation')['duration'].agg(['mean', 'std']).reset_index()

dfn['species'] = 'Nematostella'
dfn['unit'] = 'min'

dfn = dfn.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [ ]:
# Sea urchin 

dfs = pd.read_excel(f'{data_path}Raw data Sea urchin.xlsx') 

dfs['generation'] = dfs['Cycle']
dfs['std'] = add_SDs(dfs['SD mitotic'], dfs['SD intercleavage'], 0)
dfs['mean'] = dfs['Mean mitotic phase'] + dfs['Mean intercleavage']
dfs['species'] = 'Sea urchin'
dfs['unit'] = 'min'

dfs = dfs.dropna()
dfs = dfs.reindex(columns=['species', 'generation', 'mean', 'std', 'unit'])

In [21]:
dfc2 = pd.read_excel(f'{data_path}C elegans fertig richtig.xlsx')
dfc = convert_lineage_tree(dfc2, 'cleavage', 'x', 'y', 'C. elegans')


In [ ]:
# C elegans Study 2 (Lineages) 
# Create a file with a separate sheet for each individual cell-lineage of C elegans

dfc2 = pd.read_excel(f'{data_path}C elegans fertig richtig.xlsx')
#dfc2 = convert_lineage_tree(dfc2, 'cleavage', 'x', 'y', 'C. elegans')
lineages = ['AB', 'MS', 'C', 'P', 'E', 'D' ]
lineage_names = ['AB', 'MS', 'C', 'P', 'E', 'D', 'Posterior C and P', 'Posterior all']

df_list = []
for lin in lineages: 
    df = convert_lineage_tree(dfc2, 'cleavage', 'x', 'y', 'C. elegans', lineage = lin)
    df['lineage'] = lin
    df_list.append(df)



df = convert_lineage_tree(dfc2, 'cleavage', 'x', 'y', 'C. elegans')

with pd.ExcelWriter('C_elegans_lineage_paper_lineages.xlsx') as writer:

    for index, df in enumerate(df_list):
        df.to_excel(writer, sheet_name = lineage_names[index])

print('file successfully created')

lineage detected
lineage detected
lineage detected
lineage detected
lineage detected
lineage detected
file successfully created


In [27]:
with pd.ExcelWriter('Data_combined_from_raw_files.xlsx') as writer:
    dfz.to_excel(writer, sheet_name = 'Zebrafish')
    dfa1.to_excel(writer, sheet_name = 'Axolotl Dataset 1')
    dfa2.to_excel(writer, sheet_name = 'Axolotl Dataset 2')
    dfc.to_excel(writer, sheet_name = 'C elegans full')
    dfd.to_excel(writer, sheet_name = 'Dreissena')
    dfd1.to_excel(writer, sheet_name = 'Drosophila Study 1')
    dfd2.to_excel(writer, sheet_name = 'Drosophila Study 2')
    dfd3.to_excel(writer, sheet_name = 'Drosophila Study 3')
    dfn.to_excel(writer, sheet_name = 'Nematostella')
    dfs.to_excel(writer, sheet_name = 'Sea urchin')
    dfx1.to_excel(writer, sheet_name = 'Xenopus Study 1')
    dfx2.to_excel(writer, sheet_name = 'Xenopus Study 2')